In [1]:
import numpy as np
import cv2 as cv
import mediapipe as mp
import os
from matplotlib import pyplot as plt
import time
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import TensorBoard
from tensorflow.keras.models import load_model as load_model

objc[93998]: Class CaptureDelegate is implemented in both /Users/d_f_i/miniconda3/envs/tensorflow/lib/python3.10/site-packages/cv2/cv2.abi3.so (0x1647765a0) and /Users/d_f_i/miniconda3/envs/tensorflow/lib/python3.10/site-packages/mediapipe/.dylibs/libopencv_videoio.3.4.16.dylib (0x117fe4860). One of the two will be used. Which one is undefined.
objc[93998]: Class CVWindow is implemented in both /Users/d_f_i/miniconda3/envs/tensorflow/lib/python3.10/site-packages/cv2/cv2.abi3.so (0x1647765f0) and /Users/d_f_i/miniconda3/envs/tensorflow/lib/python3.10/site-packages/mediapipe/.dylibs/libopencv_highgui.3.4.16.dylib (0x107edca68). One of the two will be used. Which one is undefined.
objc[93998]: Class CVView is implemented in both /Users/d_f_i/miniconda3/envs/tensorflow/lib/python3.10/site-packages/cv2/cv2.abi3.so (0x164776618) and /Users/d_f_i/miniconda3/envs/tensorflow/lib/python3.10/site-packages/mediapipe/.dylibs/libopencv_highgui.3.4.16.dylib (0x107edca90). One of the two will be used.

In [2]:
actions = np.array(['Thank_you', 'Hello', 'Good'])
#Hello
#Thank you
#Good
#Yes
#No
#Okay
#Happy
#Sad
#Maybe


In [3]:
model_path = os.path.join('AI_models', 'actions-v0.3.4-3 signs(Thank_you, hello, good).h5')
model = load_model(model_path)

Metal device set to: Apple M1


In [4]:
def mediapipe_detection(image, model):
    image = cv.cvtColor(image, cv.COLOR_BGR2RGB)
    image.flags.writeable = False
    results = model.process(image)
    image.flags.writeable = True
    image = cv.cvtColor(image, cv.COLOR_RGB2BGR)
    return image, results

color_theme = [(16, 245, 117), (66, 125, 255), (173, 103, 100), (174, 197, 0), (210, 247, 228)]
#(16, 245, 117) green
#(255, 125, 66) Orange
#(100, 103, 173) purple
#(0, 197, 255) blue
#(0, 132, 174) dark blue
#(228, 247, 210) light green
def draw_landmarks(image, results):
    mp_drawing.draw_landmarks(image, results.face_landmarks, mp_holistic.FACEMESH_TESSELATION, mp_drawing.DrawingSpec(color=color_theme[1], thickness=1, circle_radius=1), mp_drawing.DrawingSpec(color=color_theme[0], thickness=1, circle_radius=1))
    mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS, mp_drawing.DrawingSpec(color=color_theme[3], thickness=2, circle_radius=3),mp_drawing.DrawingSpec(color=color_theme[2], thickness=1, circle_radius=1))
    mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS, mp_drawing.DrawingSpec(color=color_theme[0], thickness=2, circle_radius=3),mp_drawing.DrawingSpec(color=color_theme[2], thickness=2, circle_radius=1))
    mp_drawing.draw_landmarks(image, results. right_hand_landmarks, mp_holistic.HAND_CONNECTIONS, mp_drawing.DrawingSpec(color=color_theme[0], thickness=2, circle_radius=3),mp_drawing.DrawingSpec(color=color_theme[2], thickness=2, circle_radius=1))

def extract_keypoints(results):
    pose = np.array([[res.x, res.y, res.z, res.visibility] for res in results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(33*4)
    face = np.array([[res.x, res.y, res.z] for res in results.face_landmarks.landmark]).flatten() if results.face_landmarks else np.zeros(468*3)
    left_hand = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21*3)
    right_hand = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21*3)
    return np.concatenate([pose, face, left_hand, left_hand])

def is_hands_up(results):
    if not (results.right_hand_landmarks or results.left_hand_landmarks):
        return False
    return True
    
def render_predictions(prediction, actions, input_frame):
    output_frame = input_frame.copy()
    for num, prob in enumerate(prediction):
        cv.rectangle(output_frame, (0, 60 + num*40), (200, 90+num*40), color_theme[4], -1)
        cv.rectangle(output_frame, (0, 60 + num*40), (int(prob*200), 90+num*40), color_theme[0], -1)
        cv.putText(output_frame, actions[num], (0, 85+num*40), cv.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 0), 2, cv.LINE_AA)
    
    return output_frame

In [ ]:
last_30_frames = []
detections = []
predictions = []
threshold = 0.4

cap = cv.VideoCapture(0)
cap.set(cv.CAP_PROP_FRAME_WIDTH, 1200)
cap.set(cv.CAP_PROP_FRAME_HEIGHT, 300)

mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils

with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    frame_counter = 0
    while True:
        frame_counter += 1
        has_predicted = False
        
        ret, frame = cap.read()
        image, results = mediapipe_detection(frame, holistic)
        draw_landmarks(image, results)
        
        key_points = extract_keypoints(results)
        if is_hands_up(results):
            last_30_frames.append(key_points)
        else:
            last_30_frames = []
        last_30_frames = last_30_frames[-30:]
        #print(is_hands_up(results))
        prediction = [0, 0, 0]
        if len(last_30_frames) >= 30 and is_hands_up(results):
            prediction = model.predict(np.expand_dims(last_30_frames, axis = 0))[0]
            frame_counter = 0
            has_predicted = True
            print(actions[np.argmax(prediction)])
            predictions.append(np.argmax(prediction))
        
        image = cv.flip(image, 1)
        try:
            if np.unique(predictions[-10:])[0]==np.argmax(prediction):         
                if prediction[np.argmax(prediction)] > threshold:
                    if len(detections) > 0:
                        if actions[np.argmax(prediction)] != detections[-1]:
                            detections.append(actions[np.argmax(prediction)])
                            
                    else:
                        detections.append(actions[np.argmax(prediction)])

                    if len(detections) > 5:
                        detections = detections[-5:]
        except:
            print('empty sequence')
            
        image = render_predictions(prediction, actions, image)
        
        #cv.rectangle(image, (0, 0), (640, 40), (245, 117, 16), -1)
        #cv.putText(image, ' '.join(detections), (3, 30), cv.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 0), 2, cv.LINE_AA)
        
        cv.imshow('HHv0.3.6-AIRv2-5signs', image)
        
        if cv.waitKey(10) & 0xFF == ord('q'):
            break

    cap.release()
    cv.destroywindow('HHv0.3.6-AIRv2-5signs')

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequence
empty sequ

2023-05-11 15:54:37.099404: W tensorflow/tsl/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz


1/1 [==============================] - 1s 568ms/step
Good
1/1 [==============================] - 0s 206ms/step
Good
1/1 [==============================] - 0s 188ms/step
Good
1/1 [==============================] - 0s 193ms/step
Good
1/1 [==============================] - 0s 185ms/step
Good
1/1 [==============================] - 0s 185ms/step
Good
1/1 [==============================] - 0s 189ms/step
Good
1/1 [==============================] - 0s 198ms/step
Good
1/1 [==============================] - 0s 182ms/step
Good
1/1 [==============================] - 0s 190ms/step
Good
1/1 [==============================] - 0s 186ms/step
Good
1/1 [==============================] - 0s 196ms/step
Good
1/1 [==============================] - 0s 182ms/step
Good
1/1 [==============================] - 0s 185ms/step
Good
1/1 [==============================] - 0s 199ms/step
Good
1/1 [==============================] - 0s 184ms/step
Thank_you
1/1 [==============================] - 0s 195ms/step
Thank_you
1/1 

1/1 [==============================] - 0s 186ms/step
Good
1/1 [==============================] - 0s 200ms/step
Good
1/1 [==============================] - 0s 188ms/step
Good
1/1 [==============================] - 0s 194ms/step
Good
1/1 [==============================] - 0s 181ms/step
Good
1/1 [==============================] - 0s 182ms/step
Good
1/1 [==============================] - 0s 191ms/step
Good
1/1 [==============================] - 0s 188ms/step
Good
1/1 [==============================] - 0s 190ms/step
Good
1/1 [==============================] - 0s 179ms/step
Good
1/1 [==============================] - 0s 192ms/step
Good
1/1 [==============================] - 0s 198ms/step
Good
1/1 [==============================] - 0s 193ms/step
Good
1/1 [==============================] - 0s 198ms/step
Good
1/1 [==============================] - 0s 181ms/step
Good
1/1 [==============================] - 0s 202ms/step
Good
1/1 [==============================] - 0s 189ms/step
Good
1/1 [=========